In [6]:
import os
import pandas as pd
import numpy as np

# Подготовка аналитической базы
На этом этапе загружаем очищенные таблицы, проверяем связи между ними и создаем признаки, столбцы и все то, что будем использовать в аналитической плоскости.

### читаем файлы
Лучше читать csv, так как для моих cleaned-файлов это нормально и быстрее:
- csv читается быстрее, чем xlsx;
- меньше проблем с Excel-форматами;
- проще для дальнейшего анализа;
- Power BI тоже хорошо читает csv.

In [7]:
data_dir = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\дополнительные файлы"

contacts_path = os.path.join(data_dir, "contacts_clean.csv")
calls_path = os.path.join(data_dir, "calls_clean.csv")
deals_path = os.path.join(data_dir, "deals_clean.csv")
spend_path = os.path.join(data_dir, "spend_clean.csv")

In [8]:
contacts = pd.read_csv(contacts_path, dtype={"id": "string"})
calls = pd.read_csv(calls_path, dtype={"id": "string", "contact_id": "string"})
deals = pd.read_csv(deals_path, dtype={"id": "string", "contact_id": "string"})
spend = pd.read_csv(spend_path)

In [10]:
# После чтения CSV нужно заново привести даты:
contacts["created_time"] = pd.to_datetime(contacts["created_time"], errors="coerce")
contacts["modified_time"] = pd.to_datetime(contacts["modified_time"], errors="coerce")

calls["call_start_time"] = pd.to_datetime(calls["call_start_time"], errors="coerce")

deals["created_time"] = pd.to_datetime(deals["created_time"], errors="coerce")
deals["closing_date"] = pd.to_datetime(deals["closing_date"], errors="coerce")

spend["date"] = pd.to_datetime(spend["date"], errors="coerce")

In [11]:
# в calls есть булевый столбец:
calls["is_successful_call"] = calls["is_successful_call"].astype(bool)
# в deals есть булевый столбец:
deals["is_student"] = deals["is_student"].astype(bool)

#### Проверяем размер таблиц

In [13]:
datasets_summary = pd.DataFrame({
    "table": ["contacts", "calls", "deals", "spend"],
    "rows": [len(contacts), len(calls), len(deals), len(spend)],
    "columns": [contacts.shape[1], calls.shape[1], deals.shape[1], spend.shape[1]],
    "missing_total": [contacts.isna().sum().sum(), calls.isna().sum().sum(), deals.isna().sum().sum(), spend.isna().sum().sum()],
    "duplicates": [contacts.duplicated().sum(), calls.duplicated().sum(), deals.duplicated().sum(), spend.duplicated().sum()]})

datasets_summary

,table,rows,columns,missing_total,duplicates
0,contacts,18510,5,0,0
1,calls,92617,8,3802,0
2,deals,19824,24,85216,0
3,spend,19862,8,0,0


#### Проверяем связи между таблицами
проверяем все ли contact_id из Calls и Deals существуют в Contacts:
- Calls.contact_id  → Contacts.id
- Deals.contact_id  → Contacts.id

Берем все ID контактов из Contacts и превращаем в множество.
Берем все contact_id из Calls и вычитаем из них ID, которые есть в Contacts
Если везде 0, то все звонки и сделки с contact_id корректно связаны с Contacts.

In [14]:
contacts_ids = set(contacts["id"].dropna())

calls_unmatched_contacts = (set(calls["contact_id"].dropna()) - contacts_ids)

deals_unmatched_contacts = (set(deals["contact_id"].dropna()) - contacts_ids)

relationship_check = pd.DataFrame({
    "check": ["Calls contact_id not found in Contacts", "Deals contact_id not found in Contacts"],
    "unmatched_count": [len(calls_unmatched_contacts), len(deals_unmatched_contacts)]})

relationship_check

,check,unmatched_count
0,Calls contact_id not found in Contacts,0
1,Deals contact_id not found in Contacts,0


### Агрегируем звонки по contact_id
Это нужно, чтобы сжать таблицу звонков до уровня одного контакта. 
* Сейчас в calls одна строка = один звонок. нам для анализа сделок нужно понимать по каждому клиенту: `сколько ему звонили? сколько было успешных звонков? какая средняя длительность? когда был первый и последний звонок?`
  
* Зачем это нужно в дальнейшем? Мы можем присоединить агрегацию к Deals и посмотреть:
    - влияет ли количество звонков на покупку;
    - сколько звонков нужно до оплаты;
    - у студентов больше успешных звонков или нет;
    - какие менеджеры/источники дают клиентов, которым сложнее дозвониться;
    - есть ли связь между звонками и воронкой.

Если обобщенно, то calls_by_contact — это агрегированная таблица звонков по каждому клиенту, чтобы добавить звонковую активность к сделкам.

In [15]:
calls_by_contact = (calls.dropna(subset=["contact_id"]).groupby("contact_id").agg(
        total_calls=("id", "count"),
        successful_calls=("is_successful_call", "sum"),
        avg_call_duration_sec=("call_duration_in_seconds", "mean"),
        total_call_duration_sec=("call_duration_in_seconds", "sum"),
        first_call_time=("call_start_time", "min"),
        last_call_time=("call_start_time", "max")).reset_index())

calls_by_contact.head()

,contact_id,total_calls,successful_calls,avg_call_duration_sec,total_call_duration_sec,first_call_time,last_call_time
0,5805028000000645014,8,6,9.500000,76,2023-06-30 09:20:00,2023-07-04 15:51:00
1,5805028000000872003,12,1,0.500000,6,2023-07-05 19:19:00,2024-06-20 14:57:00
2,5805028000000939010,18,14,87.388889,1573,2023-09-13 10:58:00,2024-03-01 12:38:00
3,5805028000000942003,14,7,76.500000,1071,2023-07-06 12:00:00,2023-09-21 13:45:00
4,5805028000000961001,6,1,8.000000,48,2023-07-06 15:48:00,2023-10-05 15:33:00


### Агрегируем расходы по source и campaign
Это нужно, чтобы сжать таблицу рекламных расходов до уровня рекламного источника и кампании.
* Сейчас в spend одна строка = `рекламная статистика за конкретную дату, источник, кампанию, группу объявлений и объявление`. Нам для анализа сделок нужно понимать по каждому источнику и кампании: `сколько было показов? сколько было кликов? сколько денег потратили?`

* Зачем это нужно в дальнейшем? Мы можем использовать эту агрегацию вместе с Deals и посмотреть:
    - какие источники и кампании привели больше всего лидов;
    - какие источники и кампании дали больше студентов;
    - сколько стоил лид по каждому источнику;
    - сколько стоил студент/buyer по каждому источнику;
    - какие кампании потратили много бюджета, но дали слабый результат;
    - где рекламный бюджет используется эффективнее.
* Важно: расходы нельзя напрямую присоединять к каждой сделке строка-в-строку, потому что одна кампания может привести много сделок. Если просто присоединить расходы к каждой сделке, бюджет задвоится или затроится. Поэтому сначала мы агрегируем Spend по source и campaign, а потом используем эту таблицу для расчета рекламной эффективности.

* Если обобщенно, то spend_by_source_campaign — это агрегированная таблица рекламных расходов по источнику и кампании, чтобы позже корректно сравнить затраты с лидами, студентами и выручкой.

In [16]:
spend_by_source_campaign = (spend.groupby(["source", "campaign"], dropna=False).agg(
        total_impressions=("impressions", "sum"),
        total_clicks=("clicks", "sum"),
        total_spend=("spend", "sum")).reset_index())

spend_by_source_campaign.head()

,source,campaign,total_impressions,total_clicks,total_spend
0,Bloggers,Unknown,738460,14250,13439.00
1,CRM,Unknown,0,7995,0.00
2,Facebook Ads,01.04.23women_PL,45956,367,357.25
3,Facebook Ads,02.07.23wide_DE,594807,10281,6913.60
4,Facebook Ads,02.08.23interests_DE,5990,76,69.30


# Добавляем агрегаты звонков к Deals

In [26]:
analytics_base = deals.merge(calls_by_contact, on="contact_id", how="left")

In [18]:
call_agg_columns = [
    "total_calls",
    "successful_calls",
    "avg_call_duration_sec",
    "total_call_duration_sec"]

analytics_base[call_agg_columns] = analytics_base[call_agg_columns].fillna(0)

In [19]:
analytics_base.head()

,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,sla,content,...,contact_id,city,level_of_deutsch,is_student,total_calls,successful_calls,avg_call_duration_sec,total_call_duration_sec,first_call_time,last_call_time
0,5805028000005176025,John Doe,NaT,NaN,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,Unknown,...,5805028000000872003,Unknown,Unknown,False,12.0,1.0,0.5,6.0,2023-07-05 19:19:00,2024-06-20 14:57:00
1,5805028000005168037,John Doe,NaT,NaN,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,Unknown,...,5805028000000872003,Unknown,Unknown,False,12.0,1.0,0.5,6.0,2023-07-05 19:19:00,2024-06-20 14:57:00
2,5805028000005174051,John Doe,NaT,NaN,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,Unknown,...,5805028000000872003,Unknown,Unknown,False,12.0,1.0,0.5,6.0,2023-07-05 19:19:00,2024-06-20 14:57:00
3,5805028000005180061,John Doe,NaT,E - Non Qualified,Registered on Webinar,NaN,/workshop,web2408_DE,5422.58,Unknown,...,5805028000000872003,Unknown,Unknown,False,12.0,1.0,0.5,6.0,2023-07-05 19:19:00,2024-06-20 14:57:00
4,5805028000005176076,John Doe,NaT,NaN,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,Unknown,...,5805028000000872003,Unknown,Unknown,False,12.0,1.0,0.5,6.0,2023-07-05 19:19:00,2024-06-20 14:57:00


In [22]:
analytics_base_calls_check = pd.DataFrame({
    "metric": [
        "Всего сделок",
        "Сделок с total_calls > 0",
        "Сделок без звонков",
        "Доля сделок со звонками, %"
    ],
    "value": [
        len(analytics_base),
        (analytics_base["total_calls"] > 0).sum(),
        (analytics_base["total_calls"] == 0).sum(),
        round((analytics_base["total_calls"] > 0).mean() * 100, 2)
    ]
})

analytics_base_calls_check

,metric,value
0,Всего сделок,19824.00
1,Сделок с total_calls > 0,17241.00
2,Сделок без звонков,2583.00
3,"Доля сделок со звонками, %",86.97


In [23]:
data_merging_summary = pd.DataFrame({
    "table": [
        "contacts",
        "calls",
        "deals",
        "spend",
        "calls_by_contact",
        "spend_by_source_campaign",
        "analytics_base"
    ],
    "rows": [
        len(contacts),
        len(calls),
        len(deals),
        len(spend),
        len(calls_by_contact),
        len(spend_by_source_campaign),
        len(analytics_base)
    ],
    "columns": [
        contacts.shape[1],
        calls.shape[1],
        deals.shape[1],
        spend.shape[1],
        calls_by_contact.shape[1],
        spend_by_source_campaign.shape[1],
        analytics_base.shape[1]
    ]
})

data_merging_summary

,table,rows,columns
0,contacts,18510,5
1,calls,92617,8
2,deals,19824,24
3,spend,19862,8
4,calls_by_contact,15194,7
5,spend_by_source_campaign,66,5
6,analytics_base,19824,30


# Сохраняем аналитическую базу

In [24]:
analytics_base.to_excel(os.path.join(data_dir, "analytics_base.xlsx"), index=False)

analytics_base.to_csv(os.path.join(data_dir, "analytics_base.csv"), index=False, encoding="utf-8-sig")

calls_by_contact.to_excel(os.path.join(data_dir, "calls_by_contact.xlsx"), index=False)

spend_by_source_campaign.to_excel(os.path.join(data_dir, "spend_by_source_campaign.xlsx"), index=False)

calls_by_contact.to_csv(os.path.join(data_dir, "calls_by_contact.csv"), index=False, encoding="utf-8-sig")

spend_by_source_campaign.to_csv(os.path.join(data_dir, "spend_by_source_campaign.csv"), index=False, encoding="utf-8-sig")

data_merging_summary.to_csv(os.path.join(data_dir, "data_merging_summary.csv"), index=False, encoding="utf-8-sig")


## Вывод

На этом этапе были загружены четыре очищенные таблицы: Contacts, Calls, Deals и Spend.  
Проверены связи `Calls.contact_id → Contacts.id` и `Deals.contact_id → Contacts.id`.

Для дальнейшего анализа были созданы агрегаты звонков по `contact_id` и расходов по `source + campaign`.  
Основная аналитическая база `analytics_base` построена на основе Deals с добавлением агрегированных звонков по контакту.

Расходы из Spend не присоединялись напрямую к каждой сделке, чтобы не задвоить рекламные расходы. Для анализа рекламы используется отдельная агрегированная таблица `spend_by_source_campaign`.